# Train the beam GNN (simple)
Runtime > Change runtime type > **T4 GPU**. Then run the cells one by one, top to bottom (Shift+Enter).

**1. Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**2. Get the latest code from GitHub** (uses your GH_TOKEN secret)

In [ ]:
import os, subprocess, sys
from google.colab import userdata
token = userdata.get("GH_TOKEN")
if os.path.exists("/content/GNN-BeamRebarDetector"):
    subprocess.run(["git", "-C", "/content/GNN-BeamRebarDetector", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://{token}@github.com/yeongit5985/GNN-BeamRebarDetector.git", "/content/GNN-BeamRebarDetector"])
sys.path.insert(0, "/content/GNN-BeamRebarDetector")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdfium2", "rapidocr_onnxruntime"])
print("code ready")

**3. Your folders** (`G:\\My Drive\\Deep Learning\\GNN-BeamRebarDetector`)

In [ ]:
import json
from pathlib import Path
from datetime import datetime

BASE  = Path("/content/drive/MyDrive/Deep Learning/GNN-BeamRebarDetector")      # your working directory
JSONS = BASE / "FinalJson"      # your checked pages
PDFS  = BASE / "PDFDwg"         # the drawings
OUT   = BASE / "gnn_training" / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")   # graphs, model and charts go here
OUT.mkdir(parents=True, exist_ok=True)

pages = []
for f in sorted(JSONS.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    pages += d if isinstance(d, list) else [d]
(OUT / "all_pages.json").write_text(json.dumps(pages), encoding="utf-8")
print(len(pages), "pages found")

: 

**4. Turn each page into a graph** (a few minutes)

In [ ]:
import graph_builder as gb
gb.export_to_graphs(OUT / "all_pages.json", PDFS, OUT / "graphs")

**5. Train** on your real pages. 20% of overall pages are kept aside to test the model on drawings it has not seen. Stops early when the test score stops improving; the best version is kept.

In [ ]:
import gnn
import random

graphs = gnn.load_graphs(OUT / "graphs", verbose=False)

# Shuffle the list of graphs randomly
random.shuffle(graphs)

# Calculate 20% of the total length
test_size = int(len(graphs) * 0.2)

# Split into 20% test pages and 80% train pages
test_pages = graphs[:test_size]
train_pages = graphs[test_size:]

print("train:", len(train_pages), "pages | test:", len(test_pages), "pages")

state = {"best": -1, "bad": 0}
def watch(row):                                # early stopping: 5 checks (50 epochs) without improvement
    if "node_f1" in row:
        score = (row["node_f1"] + row["relation_ap"]) / 2
        if score > state["best"]:
            state["best"], state["bad"] = score, 0
        else:
            state["bad"] += 1

model, history, scores = gnn.train(train_pages, test_pages, epochs=200,
                                   progress=watch, should_stop=lambda: state["bad"] >= 5)
print("kept the model of epoch", scores["epoch"])

**6. Scores** (F1: 1.00 = perfect)

In [ ]:
print(gnn.metrics_text(scores))

**7. Training curves** (saved as training_curve.png)

In [ ]:
import matplotlib.pyplot as plt
ep_all  = [h["epoch"] for h in history]
checked = [h for h in history if "node_f1" in h]
ep_chk  = [h["epoch"] for h in checked]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep_all, [h["loss"] for h in history], color="#2a6fdb", linewidth=2)
ax[0].set_title("Training loss (lower = learning)"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[1].plot(ep_chk, [h["node_f1"] for h in checked], "-o", color="#2a6fdb", linewidth=2, label="Labels (what each item is)")
ax[1].plot(ep_chk, [h["relation_ap"] for h in checked], "-o", color="#e8833a", linewidth=2, label="Links (what connects to what)")
ax[1].axvline(scores["epoch"], color="gray", linestyle="--", label=f"Model kept (epoch {scores['epoch']})")
ax[1].set_ylim(0, 1); ax[1].set_title("Score on the 3 test pages (higher = better)")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Score (1.0 = perfect)"); ax[1].legend(loc="lower right")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "training_curve.png", dpi=150); plt.show()

**8. Score per type** (saved as scores_per_type.png)

In [ ]:
pc, pk = scores["per_class"], scores["per_kind"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(list(pc), [v["f1"] for v in pc.values()], color="#2a6fdb"); ax[0].set_title("Labels: score per type")
ax[1].barh(list(pk), [v["f1"] for v in pk.values()], color="#e8833a"); ax[1].set_title("Links: score per type")
for a in ax:
    a.set_xlim(0, 1); a.invert_yaxis(); a.grid(axis="x", alpha=0.3)
    for bar in a.patches:
        a.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.2f}", va="center")
plt.tight_layout(); plt.savefig(OUT / "scores_per_type.png", dpi=150); plt.show()

**9. Overfitting check**: a big gap between training and test pages = memorising

In [ ]:
train_score = gnn.evaluate(model, train_pages, model.thresholds)
print(f"Training pages: labels {train_score['node_f1']:.2f} | links {train_score['relation_f1']:.2f}")
print(f"Test pages:     labels {scores['node_f1']:.2f} | links {scores['relation_f1']:.2f}")

**10. Save the model**

In [ ]:
path = gnn.save_model(model, OUT / "model" / "beam_gnn.pt", metrics=scores,
                      trained_on=[g["name"] for g in train_pages], example=graphs[0])
print("saved:", path)